## RAG 权限检索全链路设计

---

> 本笔记承接 02_智能问答整体设计，聚焦「共享库 + 文档级/片段级权限」的完整落地方案：
> **谁（角色等级）→ 看得到什么（文档可见性 + 片段敏感等级）→ 检索时挡在哪里（SQL 层）→ 全程可审计**

### 一、整体架构（三层模型）

```
┌─────────────────────────────────────────────────────────────┐
│ ① 用户与角色层                                             │
│   admin(100) · 高级(50) · 普通(10)                         │
│   JWT 携带 role_level + is_admin                            │
│   require_admin 守卫管理接口 · 前端导航按 isAdmin 渲染      │
└───────────────────────────┬─────────────────────────────────┘
                            │ 用户提问
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ ② 智能问答层（LangGraph）                                   │
│   chat.py 组装 AgentState{question, history, user_id}      │
│   Super Router 意图识别 → route: rag                       │
│   rag_retrieve 节点：state.user_id → 查 users.role_level   │
│   → set_retrieve_scope(uid, lv)  [contextvar 注入]        │
└───────────────────────────┬─────────────────────────────────┘
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ ③ 检索层 · 权限过滤点（核心）                               │
│   LlamaIndex 自定义 VectorStore 挂载现有 pgvector 表        │
│   SQL：文档级(本人/共享达标) + 片段级(sensitivity/min_level)│
│   bge-reranker 重排 → 阈值 0.4 → 父子块回表                │
└───────────────────────────┬─────────────────────────────────┘
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ ④ 生成层 · 降级链                                          │
│   qwen-max → qwen-plus → deepseek-chat                     │
│   只基于可见内容生成 → 回答天然不含受限数据                 │
│   LangSmith 全程 Trace · 路由与降级落审计                  │
└───────────────────────────┬─────────────────────────────────┘
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ ⑤ 数据与运营层                                             │
│   documents.visibility / min_level（文档级权限）            │
│   chunks.sensitivity / min_level（片段级权限）              │
│   rule_keywords（敏感词 DB + Redis 热更新）                 │
│   audit_logs（全操作留痕）                                  │
└─────────────────────────────────────────────────────────────┘
```

**核心设计原则**：权限不是「问答之后」的检查，而是从 JWT 解析身份开始，
贯穿 AgentState → contextvar → SQL 三层，最后在**数据库层面把不可见内容挡在召回之前**——
模型根本看不到它，所以不会产生「幻觉式泄露」。


### 二、完整链路（一次提问的权限路径）

| 步骤 | 发生什么 | 权限在哪生效 |
| --- | --- | --- |
| 1. 提问 | 前端 `/api/chat/stream` → `chat.py` | — |
| 2. 组装状态 | `AgentState{question, history, session_id, user_id}` | user_id 由 JWT 解析，贯穿全程 |
| 3. 意图路由 | LangGraph Super Router → `route: "rag"` | 无权限语义，只管去向 |
| 4. 检索注入 | `rag_retrieve` 用 state.user_id 查 role_level → `set_retrieve_scope(uid, lv)` | **contextvar 每请求协程隔离** |
| 5. SQL 过滤 | `ChunksVectorStore.query` 拼接权限条件 | **文档级 + 片段级双重过滤（核心）** |
| 6. 重排阈值 | bge-reranker 重排 → 0.4 阈值 → 父子块回表 | 只返回可见块 |
| 7. 生成 | 降级链 LLM 基于可见上下文作答 | 回答天然不含受限数据 |
| 8. 留痕 | 路由、降级、上下文写入日志/审计表 | — |

---

### 三、技术栈

| 层 | 选型 | 职责 |
| --- | --- | --- |
| 前端 | React 18 + Vite + Ant Design | 导航按角色渲染、文档权限管理、敏感词管理、用户管理、审计查询 |
| 后端 | FastAPI + SQLAlchemy | 路由、权限依赖（require_level / require_admin）、管理接口 |
| 存储 | PostgreSQL(pgvector) + Redis | 向量检索 + 敏感词热更新缓存 |
| AI 框架 | LangGraph（图编排/循环）+ LlamaIndex（检索） | Super Router 意图路由 + 自定义 VectorStore |
| 模型 | BGE-M3（embedding）+ bge-reranker + Qwen 降级链 | 向量化 / 重排 / 生成 |
| 可观测 | LangSmith + 日志审计表 | 全链路 Trace + 审计留痕 |


### 四、关键代码

#### 1. 权限过滤 SQL —— 整个方案的心脏（`backend/app/agent/retriever.py`）

```python
def _perm_filter_sql() -> str:
    """拼权限过滤 SQL：文档级（本人 或 共享达标）+ 片段级（sensitivity/min_level）"""
    scope = _retrieve_scope.get()
    uid = scope.get("user_id")
    lv = scope.get("role_level") or 10
    doc_cond = "(d.user_id = :perm_uid OR (d.visibility <> 'restricted' AND d.min_level <= :perm_lv))" if uid else \
               "(d.visibility <> 'restricted' AND d.min_level <= :perm_lv)"
    return f"""AND ({doc_cond})
               AND (c.sensitivity IS NULL OR c.sensitivity = 'public' OR c.min_level <= :perm_lv)"""


def _perm_params() -> dict:
    scope = _retrieve_scope.get()
    return {"perm_uid": scope.get("user_id"), "perm_lv": scope.get("role_level") or 10}
```

查询时拼接（pgvector 余弦）：

```sql
SELECT c.id, c.parent_chunk_id, c.doc_id, c.content, d.title
FROM chunks c
JOIN documents d ON c.doc_id = d.id
WHERE c.is_deleted = FALSE
  -- 文档级：本人文档 或（非受限 且 等级达标）
  AND (d.user_id = :perm_uid OR (d.visibility <> 'restricted' AND d.min_level <= :perm_lv))
  -- 片段级：公开 或 等级达标
  AND (c.sensitivity IS NULL OR c.sensitivity = 'public' OR c.min_level <= :perm_lv)
ORDER BY c.embedding <=> :vec LIMIT :k
```

#### 2. 权限上下文注入（contextvar，每请求隔离，SSE 流式也安全）

```python
_retrieve_scope: ContextVar[dict] = ContextVar("retrieve_scope", default={"user_id": None, "role_level": 10})


def set_retrieve_scope(user_id: str | None, role_level: int = 10) -> None:
    """在 chat 请求入口设置检索权限 scope（FastAPI 每请求协程隔离）"""
    _retrieve_scope.set({"user_id": user_id, "role_level": role_level or 10})


def rag_retrieve(state, ...):            # LangGraph 节点
    user = db.query(User).filter(User.id == state["user_id"]).first()
    set_retrieve_scope(user.id, user.role_level)   # ← 从状态取 user_id，注入检索 scope
```

#### 3. chat 链路把 user_id 传进图（权限的源头）

```python
state: AgentState = {
    "question": req.question,
    "history": history_messages,
    "session_id": session_id,
    "user_id": user_id,        # ← 关键：问答链路全程携带用户身份
}
```

#### 4. 管理接口权限依赖（`backend/app/api/auth.py`）

```python
def require_level(min_level: int = 100):
    """权限依赖工厂：当前用户 role_level >= min_level 才放行，否则 403"""
    def _dep(user: User = Depends(get_current_user)) -> User:
        if (user.role_level or 10) < min_level:
            raise HTTPException(status_code=403, detail="权限不足：需要更高权限等级")
        return user
    return _dep


require_admin = require_level(100)   # 上传 / 权限 / 敏感词 / 用户 / 审计 全部挂这个
```


### 五、敏感打标管线（上传时自动分类）

```
上传文档(仅 admin)
    ↓
┌────────────────────────────────────────────┐
│ 三段式打标（优先级从高到低）                │
│ 1. 标记语法  <!-- sensitive:secret -->     │  ← 文档作者显式标记，最高优先
│ 2. 敏感词命中 rule_keywords（DB+Redis）    │  ← 关键词粗筛，不进 LLM，零成本
│ 3. LLM 兜底分类（降级链）                  │  ← 前面都没命中才调模型
└────────────────────────────────────────────┘
    ↓
chunks.sensitivity = public / internal / secret
chunks.min_level = 10 / 50 / 100
chunks.classify_reason = 标记依据（可追溯）
    ↓
fail-safe：LLM 失败 → 宁严勿松，默认标 secret
```

**运营配置**：敏感词/分类关键词存数据库表 `rule_keywords`，Redis 缓存热更新——
后台加词 → 写 DB → 删缓存 → 下个请求自动生效，**无需重启服务**。
（明确否决 YAML 做运营配置，YAML 只留给代码级静态参数。）

**人工复核**：admin 可在「敏感片段复核」弹窗逐块改标（如 LLM 把含人事制度的大块判为 internal，
可人工修正为 secret 或拆块），改标即写审计。

---

### 六、验证结果（端到端实测）

| 验证项 | 结果 |
| --- | --- |
| 注册新用户 | 默认 role_level=10，is_admin=false |
| 上传自动打标 | "薪酬"关键词命中 → secret（fail-safe 链路正常） |
| 文档级权限 | restricted/100 对普通用户不可见（列表过滤 + 预览 403）；恢复 public/10 后可见 |
| 管理权限 | 普通用户上传 / 设权限 / 复核 / 敏感词 → 全部 403 |
| 检索层过滤 | level10 检索不到 secret 薪酬块；level100 可检索到（精确内容检索） |
| LLM 分类 | 公司简介→public、技术支持→internal、薪酬→secret（关键词命中） |
| chat 端到端 | 普通用户问"薪酬标准"→ 检索为空、回答不引用库内敏感数据（无泄露） |
| 审计日志 | upload / set_permission / set_role / rename / 启停账号 → 全部留痕 |

**一句话总结**：权限从 JWT 解析身份开始，贯穿 AgentState → contextvar → SQL 三层，
最终在数据库层面把不可见内容挡在召回之前——模型根本看不到受限数据，从根上杜绝幻觉式泄露。
